In [ ]:
import os
import requests
import json
from dotenv import load_dotenv
import yfinance as yf
from yahooquery import Ticker
from groq import Groq

load_dotenv()
serper_api_key = os.getenv("SERP_API_KEY")
api_key_groq = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key_groq)


# ---------------------------------------------------------------------------
# Data-collection helpers
# ---------------------------------------------------------------------------

def get_company_news(company_name: str) -> list | None:
    """Fetch recent news for *company_name* via the Serper API."""
    headers = {
        "X-API-KEY": serper_api_key,
        "Content-Type": "application/json",
    }
    payload = {"q": company_name}

    try:
        response = requests.post(
            "https://google.serper.dev/news", headers=headers, json=payload, timeout=15
        )
        response.raise_for_status()
        data = response.json()
        return data.get("news")
    except requests.RequestException as e:
        print(f"[get_company_news] HTTP error: {e}")
        return None


def write_news_to_file(news: list, filename: str) -> None:
    """Write a list of news items to *filename* (overwrites existing content)."""
    with open(filename, "w", encoding="utf-8") as file:
        for item in news:
            if item is not None:
                title = item.get("title", "No title")
                link = item.get("link", "No link")
                date = item.get("date", "No date")
                file.write(f"Title: {title}\n")
                file.write(f"Link:  {link}\n")
                file.write(f"Date:  {date}\n\n")


def get_stock_evolution(company_ticker: str, period: str = "1y", filename: str = "investment.txt"):
    """Fetch historical price data and append it to *filename*."""
    stock = yf.Ticker(company_ticker)
    try:
        hist = stock.history(period=period)
    except Exception as e:
        print(f"[get_stock_evolution] Error fetching history: {e}")
        return None

    with open(filename, "a", encoding="utf-8") as file:
        file.write(f"\nStock Evolution for {company_ticker}:\n")
        file.write(hist.to_string())
        file.write("\n")

    return hist


def get_financial_statements(ticker: str, filename: str = "investment.txt") -> None:
    """Fetch balance sheet, cash flow, income statement and valuation measures,
    then append them to *filename*."""
    company = Ticker(ticker)

    # -- Balance sheet --
    try:
        balance_sheet_data = company.balance_sheet()
        balance_sheet = (
            balance_sheet_data.to_string()
            if hasattr(balance_sheet_data, "to_string")
            else str(balance_sheet_data)
        )
    except Exception as e:
        print(f"[get_financial_statements] Balance sheet error: {e}")
        balance_sheet = "N/A"

    # -- Cash flow --
    try:
        cash_flow_data = company.cash_flow(trailing=False)
        cash_flow = (
            cash_flow_data.to_string()
            if hasattr(cash_flow_data, "to_string")
            else str(cash_flow_data)
        )
    except Exception as e:
        print(f"[get_financial_statements] Cash flow error: {e}")
        cash_flow = "N/A"

    # -- Income statement --
    try:
        income_statement_data = company.income_statement()
        income_statement = (
            income_statement_data.to_string()
            if hasattr(income_statement_data, "to_string")
            else str(income_statement_data)
        )
    except Exception as e:
        print(f"[get_financial_statements] Income statement error: {e}")
        income_statement = "N/A"

    # -- Valuation measures --
    try:
        valuation_measures = str(company.valuation_measures)
    except Exception as e:
        print(f"[get_financial_statements] Valuation measures error: {e}")
        valuation_measures = "N/A"

    with open(filename, "a", encoding="utf-8") as file:
        file.write("\nBalance Sheet\n")
        file.write(balance_sheet)
        file.write("\nCash Flow\n")
        file.write(cash_flow)
        file.write("\nIncome Statement\n")
        file.write(income_statement)
        file.write("\nValuation Measures\n")
        file.write(valuation_measures)


def get_data(
    company_name: str,
    company_ticker: str,
    period: str = "1y",
    filename: str = "investment.txt",
):
    """Orchestrate data collection: news → stock evolution → financials.

    The file is *overwritten* (via write_news_to_file) so each run starts
    fresh; stock / financial data is then appended to that same file.
    """
    # News (overwrites the file so we start clean on every run)
    news = get_company_news(company_name)
    if news:
        write_news_to_file(news, filename)
    else:
        print("[get_data] No news found; starting file empty.")
        # Ensure the file exists and is empty so appends below work cleanly
        open(filename, "w").close()

    # Stock price history — pass filename so it uses the same file
    hist = get_stock_evolution(company_ticker, period=period, filename=filename)

    # Financial statements — pass filename so it uses the same file
    get_financial_statements(company_ticker, filename=filename)

    return hist


# ---------------------------------------------------------------------------
# Token-safe summarization
# ---------------------------------------------------------------------------

# Safe character budget per chunk — well under 6000 TPM for llama-3.1-8b-instant.
# ~4 chars ≈ 1 token, so 3500 chars ≈ ~875 tokens, leaving headroom for the prompt.
_CHUNK_CHARS = 3_500
_SUMMARY_CHARS = 3_000   # max chars to keep from the final combined summary


def summarize_in_chunks(raw_text: str, company_name: str, ticker: str) -> str:
    """
    Split *raw_text* into chunks of ~_CHUNK_CHARS characters and ask the LLM
    to distil each chunk into key financial figures and facts.  The resulting
    mini-summaries are concatenated and, if still too long, trimmed to
    _SUMMARY_CHARS so the thesis prompt stays within the TPM limit.
    """
    chunks = [
        raw_text[i : i + _CHUNK_CHARS]
        for i in range(0, len(raw_text), _CHUNK_CHARS)
    ]

    mini_summaries: list[str] = []

    for idx, chunk in enumerate(chunks, 1):
        print(f"[summarize_in_chunks] Summarising chunk {idx}/{len(chunks)} ...")
        try:
            resp = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                max_tokens=400,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a financial data extractor. "
                            "Extract and list only the key numerical figures, "
                            "ratios, and factual highlights from the text. "
                            "Be concise — bullet points only, no prose."
                        ),
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Extract key financial data for {company_name} ({ticker}) "
                            f"from the following snippet:\n\n{chunk}"
                        ),
                    },
                ],
            )
            mini_summaries.append(resp.choices[0].message.content.strip())
        except Exception as e:
            print(f"[summarize_in_chunks] Chunk {idx} summarisation failed: {e}")

    combined = "\n\n".join(mini_summaries)
    # Final safety trim
    return combined[:_SUMMARY_CHARS]


# ---------------------------------------------------------------------------
# Main agent entry-point
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """
You are a skeptical, independent equity research analyst
with no affiliation to the company you are analysing.
Your job is to give an honest, unbiased assessment.
Treat BUY, HOLD, and SELL as equally valid conclusions.
Do NOT default to BUY — only recommend it if the data
strongly justifies it across valuation, growth, profitability,
and risk. If valuation is stretched, debt is high, or growth
is slowing, say HOLD or SELL and explain why clearly.

Write a detailed Markdown investment thesis suitable for
rendering in a Streamlit UI.

Use this exact structure:

# [Company Name] — Investment Thesis

## 1. Company Overview
3 to 5 sentences covering business model, segments, and market position.

## 2. Recent News
- Bullet point per news item with date and key detail

## 3. Revenue Analysis
Provide a Markdown table with the last 3 years of data:

| Year | Revenue | YoY Growth |
|------|---------|------------|
| YYYY | $XXXb   | XX%        |
| YYYY | $XXXb   | XX%        |
| YYYY | $XXXb   | XX%        |

Follow the table with 2 to 3 sentences of commentary.

## 4. Profitability Analysis
Provide a Markdown table with the last 3 years of data:

| Year | Net Income | Net Margin | Gross Margin | Operating Margin |
|------|------------|------------|--------------|-----------------|
| YYYY | $XXXb      | XX%        | XX%          | XX%             |
| YYYY | $XXXb      | XX%        | XX%          | XX%             |
| YYYY | $XXXb      | XX%        | XX%          | XX%             |

Follow the table with 2 to 3 sentences of commentary.

## 5. Balance Sheet Analysis
Provide a Markdown table with the last 3 years of data:

| Year | Total Assets | Total Debt | Cash & Equivalents | Debt-to-Equity |
|------|-------------|------------|-------------------|----------------|
| YYYY | $XXXb       | $XXXb      | $XXXb             | X.Xx           |
| YYYY | $XXXb       | $XXXb      | $XXXb             | X.Xx           |
| YYYY | $XXXb       | $XXXb      | $XXXb             | X.Xx           |

Follow the table with 2 to 3 sentences of commentary.

## 6. Cash Flow Analysis
Provide a Markdown table with the last 3 years of data:

| Year | Operating Cash Flow | Free Cash Flow | FCF Margin |
|------|--------------------|--------------  |------------|
| YYYY | $XXXb              | $XXXb          | XX%        |
| YYYY | $XXXb              | $XXXb          | XX%        |
| YYYY | $XXXb              | $XXXb          | XX%        |

Follow the table with 2 to 3 sentences of commentary.

## 7. Valuation Analysis
Provide a Markdown table with the last 3 years of data:

| Metric              | YYYY  | YYYY  | YYYY  |
|---------------------|-------|-------|-------|
| P/E Ratio           | XXx   | XXx   | XXx   |
| EV/Sales            | XXx   | XXx   | XXx   |
| Price-to-Book       | XXx   | XXx   | XXx   |
| EV/EBITDA           | XXx   | XXx   | XXx   |

Follow the table with 2 to 3 sentences on whether current
multiples are attractive, fair, or stretched vs peers.

## 8. Risks
- Use bullet points for each risk, one sentence per risk

## 9. Bull Case
- Use bullet points for bull arguments
- **Price Target (Bull):** $XXX — state the assumed multiple and why

## 10. Bear Case
- Use bullet points for bear arguments
- **Price Target (Bear):** $XXX — state the assumed multiple and why

## 11. Short-Term Outlook
3 to 5 sentences covering the next 1 to 2 quarters.

## 12. Long-Term Outlook
3 to 5 sentences covering the next 3 to 5 years.

## 13. Final Recommendation

Before the rating, score the company on each dimension:

| Dimension      | Score  | Reasoning          |
|----------------|--------|--------------------|
| Revenue Growth | X / 10 | one line reasoning |
| Profitability  | X / 10 | one line reasoning |
| Balance Sheet  | X / 10 | one line reasoning |
| Cash Flow      | X / 10 | one line reasoning |
| Valuation      | X / 10 | one line reasoning |
| Risk Level     | X / 10 | 10 = very high risk|
| **Composite**  | X / 10 |                    |

Scoring guide — your final rating MUST match the composite:
- Composite 8 to 10 → BUY
- Composite 5 to 7  → HOLD
- Composite 1 to 4  → SELL

Do not let the final rating contradict the composite score.

Rules:
- Use specific numbers and percentages wherever available
- Bold all key metrics using **value**
- Never use plain paragraphs where a table fits better
- Do not wrap output in code blocks or HTML tags
- Tables must use proper Markdown pipe syntax

Conclude with the rating on its own line:

BUY
HOLD
or
SELL
"""

GET_DATA_FUNCTION = {
    "name": "get_data",
    "description": "Get financial data on a specific company for investment purposes",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {
                "type": "string",
                "description": "The name of the company",
            },
            "company_ticker": {
                "type": "string",
                "description": "The ticker symbol of the company's stock",
            },
            "period": {
                "type": "string",
                "description": "The historical period for stock data (e.g. '1y', '6mo')",
            },
            "filename": {
                "type": "string",
                "description": "The filename used to store collected data",
            },
        },
        "required": ["company_name", "company_ticker"],
    },
}


def financial_analyst(request: str) -> tuple[str, object] | None:
    """
    1. Ask the LLM to extract company name + ticker from *request*.
    2. Collect financial data via get_data().
    3. Ask the LLM to produce a full investment thesis from the collected data.

    Returns (thesis_html, hist_dataframe) or None on failure.
    """
    print(f"[financial_analyst] Request: {request}")

    # --- Step 1: extract company name and ticker via function-calling ---
    first_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": (
                    f"Given the user request, what is the company name and the "
                    f"company stock ticker?: {request}"
                ),
            }
        ],
        functions=[GET_DATA_FUNCTION],
        function_call={"name": "get_data"},
    )

    message = first_response.choices[0].message

    # Guard: make sure the model actually returned a function call
    if not message.function_call:
        print("[financial_analyst] No function call returned by the model.")
        return None

    # FIX: use attribute access, not dictionary subscript
    try:
        arguments = json.loads(message.function_call.arguments)
    except json.JSONDecodeError as e:
        print(f"[financial_analyst] Failed to parse function arguments: {e}")
        return None

    company_name = arguments.get("company_name")
    company_ticker = arguments.get("company_ticker")
    period = arguments.get("period") or "1y"
    filename = arguments.get("filename") or "investment.txt"

    if not company_name or not company_ticker:
        print("[financial_analyst] Missing company_name or company_ticker in arguments.")
        return None

    print(f"[financial_analyst] Company: {company_name} | Ticker: {company_ticker}")

    # --- Step 2: collect data ---
    hist = get_data(company_name, company_ticker, period=period, filename=filename)

    # --- Step 3: read and summarize collected data in chunks ---
    try:
        with open(filename, "r", encoding="utf-8") as file:
            raw_content = file.read()
    except FileNotFoundError:
        print(f"[financial_analyst] Data file '{filename}' not found.")
        return None

    # Summarize the raw data in chunks to stay within TPM limits
    content_summary = summarize_in_chunks(raw_content, company_name, company_ticker)

    # --- Step 4: generate investment thesis ---
    second_response = client.chat.completions.create(
        #model="llama-3.1-8b-instant",
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    f"{request}\n\nHere is a summary of the financial data for "
                    f"{company_name} ({company_ticker}):\n\n{content_summary}\n\n"
                    "Based on the data above, write the full investment thesis in HTML."
                ),
            },
        ],
    )

    thesis_html = second_response.choices[0].message.content
    return thesis_html, hist
# ---------------------------------------------------------------------------
# Quick smoke-test
# ---------------------------------------------------------------------------
# Safe version — only runs when you execute the notebook directly,
# never when imported by app.py
RUN_TEST = False  # ← Set to True only when testing directly in the notebook
                  #   Keep False when running via app.py (prevents import crash)

if RUN_TEST:
    result = financial_analyst("Should I invest in Tesla?")
    if result is not None:
        thesis, hist = result
        print("\n--- THESIS (first 500 chars) ---")
        print(thesis[:500])
    else:
        print("Analysis failed.")


[financial_analyst] Request: Should I invest in ServiceNow?
[financial_analyst] Company: ServiceNow | Ticker: NOW
[summarize_in_chunks] Summarising chunk 1/22 ...
[summarize_in_chunks] Summarising chunk 2/22 ...
[summarize_in_chunks] Summarising chunk 3/22 ...
[summarize_in_chunks] Summarising chunk 4/22 ...
[summarize_in_chunks] Summarising chunk 5/22 ...
[summarize_in_chunks] Summarising chunk 6/22 ...
[summarize_in_chunks] Summarising chunk 7/22 ...
[summarize_in_chunks] Summarising chunk 8/22 ...
[summarize_in_chunks] Summarising chunk 9/22 ...
[summarize_in_chunks] Summarising chunk 10/22 ...
[summarize_in_chunks] Summarising chunk 11/22 ...
[summarize_in_chunks] Summarising chunk 12/22 ...
[summarize_in_chunks] Summarising chunk 13/22 ...
[summarize_in_chunks] Summarising chunk 14/22 ...
[summarize_in_chunks] Summarising chunk 15/22 ...
[summarize_in_chunks] Summarising chunk 16/22 ...
[summarize_in_chunks] Summarising chunk 17/22 ...
[summarize_in_chunks] Summarising chunk 18/22

In [ ]:
#print(thesis)


# ServiceNow — Investment Thesis

## 1. Company Overview
ServiceNow is a cloud-based platform that provides a range of services, including IT service management, customer service management, and security operations. The company operates in the software industry and is known for its innovative approach to service management. ServiceNow has a strong market position, with a large customer base and a wide range of products and services. The company has made strategic acquisitions, such as the acquisition of ai.work, to expand its capabilities and enhance its offerings. ServiceNow operates in a highly competitive industry, but its unique approach and strong brand have enabled it to establish a significant presence in the market.

## 2. Recent News
- ServiceNow acquired ai.work for tens of millions, expanding its capabilities in artificial intelligence.
- Guggenheim upgraded the stock, seeing 45%+ upside potential.
- Jim Cramer considers the gains in ServiceNow's stock to be ephemeral.

## 3

In [ ]:
#print(thesis)

# Tesla — Investment Thesis
## 1. Company Overview
Tesla is a leading electric vehicle (EV) and clean energy company that designs, manufactures, and sells EVs, solar panels, and energy storage systems. The company has a strong market position in the EV industry, with a brand valued at over $100 billion. Tesla's business segments include automotive sales, energy generation and storage, and services. The company's mission is to accelerate the world's transition to sustainable energy.

## 2. Recent News
- Tesla expands robotaxi services to a third state, increasing its presence in the autonomous vehicle market.
- Tesla's stock declines 7% due to a hacking incident, marking its worst day in nearly a year.
- Tesla experiences a 25% increase in sales due to European recovery, indicating a strong demand for its products.
- Tesla sets a quarterly deliveries record, showcasing its operational efficiency.

## 3. Revenue Analysis
| Year | Revenue | YoY Growth |
|------|---------|------------|
| 2

In [ ]:
#print(thesis)

<h1>Investment Thesis: NVIDIA (NVDA)</h1>

<h2>1. Company Overview</h2>
<p>NVIDIA is a leader in the technology industry, specializing in the design and manufacture of graphics processing units (GPUs) and high-performance computing hardware. The company has risen to the top of a $10 billion market and has established itself as a key player in the AI and gaming industries.</p>

<h2>2. Recent News</h2>
<p>Recent news includes the development of a 6G ready digital twin with Samsung for a Japan telecom giant. Additionally, there have been reports of smuggled NVIDIA AI servers in China, with a cost of $1 million. The company's cooling system has also been noted to operate at a temperature of 113°F.</p>

<h2>3. Revenue Analysis</h2>
<p>NVIDIA's revenue has shown significant growth, with a 47.5% increase in 2023 compared to 2022. The company's current revenue for 2022 is $26.97 billion, and for 2023, it is $39.73 billion. This growth is a testament to the company's strong position in the mark

In [ ]:
#print(thesis)

# ServiceNow — Investment Thesis

## 1. Company Overview
ServiceNow is a leading provider of cloud-based solutions that automate and manage IT services for enterprises. The company's platform provides a range of tools for IT service management, customer service management, and security operations. With a strong market position and a growing customer base, ServiceNow has established itself as a major player in the IT service management industry. The company's business model is primarily based on subscription revenues, with a focus on delivering high-quality customer experiences and driving innovation through its platform. ServiceNow operates in a highly competitive market, but its unique approach to IT service management has helped it to differentiate itself from its peers.

## 2. Recent News
- Guggenheim upgraded ServiceNow from Neutral to Buy on [date not available], citing the company's strong growth potential and improving profitability.
- ServiceNow's stock price plummeted 50% in o

In [ ]:
#print(thesis)

<h1>Investment Thesis: Google (GOOGL)</h1>

<h2>1. Company Overview</h2>
<p>Google, a subsidiary of Alphabet Inc., is a multinational technology company that specializes in Internet-related services and products. The company's primary revenue sources include advertising, cloud computing, and hardware sales. Google's innovative products, such as its search engine, YouTube, and Android operating system, have made it a dominant player in the technology industry.</p>

<h2>2. Recent News</h2>
<p>Google has recently faced antitrust fines, including a €4.1 billion fine by the EU and $1.5 billion in antitrust damages ordered to be paid to Klarna by a Swedish court. Additionally, YouTube's worth has been estimated to be around $550 billion, highlighting the significant value of Google's subsidiaries.</p>

<h2>3. Revenue Analysis</h2>
<p>Unfortunately, the provided data does not include Google's revenue figures. However, according to publicly available data, Google's revenue has been consistentl